# 01. qforte Molecule And Jordan-Wigner Blocks

This notebook is the qforte/OpenFermion side of the workflow. Run it in the `qfe_env_v1` environment. It creates a notebook-local molecule folder, builds the qforte molecule, groups the molecular Hamiltonian into Hermitian pairs, ports those groups to OpenFermion `FermionOperator` records, then applies OpenFermion's Jordan-Wigner transform block by block.

The important handoff file is `workflow_manifest.json`. The Qiskit notebook reads that manifest instead of asking you to remember every intermediate path.

## Cell 1: Define Molecule And Build qforte Object

All user-facing molecule options live in this cell. The helper automatically builds `molecule_name` from the geometry and basis unless `MOLECULE_LABEL_OVERRIDE` is provided.

In [2]:
from pathlib import Path
import sys

CANDIDATE_ROOTS = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next(path for path in CANDIDATE_ROOTS if (path / "notebooks" / "workflow_helpers").exists())
NOTEBOOKS_ROOT = REPO_ROOT / "notebooks"
if str(NOTEBOOKS_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_ROOT))

from workflow_helpers.qforte_export_helpers import build_qforte_molecule_and_metadata

# --- User-facing molecule options ---
QFORTE_SOURCE_ROOT = Path("/Users/nstair/Src/my_qforte/qforte")
BASIS = "sto-3g"
CHARGE = 0
MULTIPLICITY = 1
SYMMETRY = "c1"
REQUESTED_FCI_ROOTS = 2
RUN_FCI = True
MOLECULE_LABEL_OVERRIDE = None  # Example: "linear_h4_sto3g"; None means auto-generate.

# Linear H4 example. Change this geometry and rerun the whole notebook to make a new molecule folder.
GEOMETRY = [
    ("H", (0.0, 0.0, 1.0)),
    ("H", (0.0, 0.0, 2.0)),
    # ("H", (0.0, 0.0, 3.0)),
    # ("H", (0.0, 0.0, 4.0)),
]

cell1 = build_qforte_molecule_and_metadata(
    notebooks_root=NOTEBOOKS_ROOT,
    qforte_source_root=QFORTE_SOURCE_ROOT,
    geometry=GEOMETRY,
    basis=BASIS,
    charge=CHARGE,
    multiplicity=MULTIPLICITY,
    symmetry=SYMMETRY,
    requested_fci_roots=REQUESTED_FCI_ROOTS,
    molecule_label=MOLECULE_LABEL_OVERRIDE,
    run_fci=RUN_FCI,
)

qforte = cell1["qforte"]
mol = cell1["molecule"]
molecule_name = cell1["molecule_name"]
paths = cell1["paths"]
molecule_metadata = cell1["metadata"]
manifest_path = paths["manifest_json"]

print(f"\nCopy this molecule_name into notebook 2: {molecule_name}")

 ==> Psi4 geometry <==
-------------------------
0  1
H  0.0  0.0  1.0
H  0.0  0.0  2.0
symmetry c1
units angstrom

  FCI Eigenstate Energies
======================================:
  i: 0  Ei:       -1.1011503301
  i: 1  Ei:       -0.3522906263

Cell 1 Output: qforte Molecule
Molecule name:                         diatomic_h2_sto_3g
Workflow root:                         /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g
Metadata JSON:                         /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/metadata/diatomic_h2_sto_3g_metadata.json
Manifest JSON:                         /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/metadata/workflow_manifest.json
Qubits:                                4
Electrons:                             2
HF energy:                             -1.066108649185
FCI root 0:                            -1.101150330133
FCI status:                            all_requested_roots_returned

Copy this molecule_name into 

## Cell 2: Build Hermitian Pairs

qforte groups the second-quantized Hamiltonian into Hermitian-pair blocks. These are the grouped objects that later become grouped Pauli evolution circuits.

In [3]:
from workflow_helpers.qforte_export_helpers import build_and_save_hermitian_pairs

# --- User-facing Hermitian-pair options ---
OUTER_POOL_COEFFICIENT = 1.0

hermitian_pair_archive = build_and_save_hermitian_pairs(
    qforte=qforte,
    molecule=mol,
    molecule_metadata=molecule_metadata,
    output_json=paths["hermitian_pairs_json"],
    manifest_json=manifest_path,
    outer_pool_coefficient=OUTER_POOL_COEFFICIENT,
)


Cell 2 Output: Hermitian Pairs
Hermitian-pair JSON:                   /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/hamiltonian_blocks/diatomic_h2_sto_3g_hermitian_pairs.json
Groups:                                13
Scalar groups:                         1
Nontrivial groups:                     12
Scalar energy:                         +0.529177210670+0.000000000000j


## Cell 3: Port Blocks To OpenFermion FermionOperator Records

This cell does not perform Jordan-Wigner yet. It writes a readable record of each grouped `FermionOperator` so the fermion-to-qubit step is clearly separated from qforte's grouping step.

In [4]:
from workflow_helpers.qforte_export_helpers import build_and_save_openfermion_blocks

# --- User-facing OpenFermion conversion options ---
COEFFICIENT_TOLERANCE = 1.0e-12

fermion_block_archive = build_and_save_openfermion_blocks(
    hermitian_pair_archive=hermitian_pair_archive,
    output_json=paths["fermion_blocks_json"],
    manifest_json=manifest_path,
    coefficient_tolerance=COEFFICIENT_TOLERANCE,
)


Cell 3 Output: OpenFermion Blocks
Fermion block JSON:                    /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/hamiltonian_blocks/diatomic_h2_sto_3g_fermion_blocks.json
Groups:                                13
Compressed terms:                      15


## Cell 4: Apply OpenFermion Jordan-Wigner Transform

Each grouped fermionic block is mapped to Pauli strings using OpenFermion's Jordan-Wigner transform. The output JSON is Qiskit-ready and becomes the input to notebook 2.

In [5]:
from workflow_helpers.qforte_export_helpers import build_and_save_grouped_paulis

# --- User-facing Jordan-Wigner options ---
JW_COEFFICIENT_TOLERANCE = 1.0e-12

grouped_pauli_archive = build_and_save_grouped_paulis(
    fermion_block_archive=fermion_block_archive,
    output_json=paths["grouped_paulis_json"],
    manifest_json=manifest_path,
    coefficient_tolerance=JW_COEFFICIENT_TOLERANCE,
)

print("\nNotebook 1 complete. Notebook 2 can now read:")
print(f"  molecule_name = {molecule_name!r}")
print(f"  manifest      = {manifest_path}")


Cell 4 Output: Jordan-Wigner Grouped Pauli Blocks
Grouped Pauli JSON:                    /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/hamiltonian_blocks/diatomic_h2_sto_3g_grouped_paulis.json
Groups:                                13
Total Pauli terms:                     49
Identity Pauli terms:                  11
Non-identity Pauli terms:              38
Manifest JSON:                         /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/metadata/workflow_manifest.json

Notebook 1 complete. Notebook 2 can now read:
  molecule_name = 'diatomic_h2_sto_3g'
  manifest      = /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/metadata/workflow_manifest.json
